### Option 1: TED talks via Huggingface

#### Load Dataset

In [22]:
from datasets import load_dataset
import soundfile as sf
import numpy as np
import re
import os

# Load TEDLIUM in streaming mode
tedlium_dataset = load_dataset(
    "LIUM/tedlium",
    name="release1",
    split="test",
    streaming=True
)

#### Get samples

In [ ]:
def clean_transcript(text):
    if "ignore_time_segment_in_scoring" in text:
        return ""
    return re.sub(r"\b(\w)\s+'(\w)", r"\1\2", text).strip()

def save_n_samples(tedlium_dataset, n=10, duration_sec=120, out_folder="data"):
    os.makedirs(out_folder, exist_ok=True)
    sample_iter = iter(tedlium_dataset)
    for count in range(n):
        audio_arrays, transcripts = [], []
        total_samples, sampling_rate = 0, None
        while total_samples < duration_sec * (sampling_rate if sampling_rate else 16000):
            try:
                sample = next(sample_iter)
            except StopIteration:
                break
            waveform = sample["audio"]["array"]
            sr = sample["audio"]["sampling_rate"]
            text = clean_transcript(sample["text"])
            if not text:
                continue
            if sampling_rate is None:
                sampling_rate = sr
            elif sr != sampling_rate:
                raise ValueError("Sample rate mismatch!")
            audio_arrays.append(waveform)
            transcripts.append(text)
            total_samples += len(waveform)
        if not audio_arrays:
            break
        full_audio = np.concatenate(audio_arrays)[:duration_sec * sampling_rate]
        full_transcript = " ".join(transcripts)
        sf.write(os.path.join(out_folder, f"ted_sample_{count+1}.wav"), full_audio, samplerate=sampling_rate)
        with open(os.path.join(out_folder, f"ted_sample_{count+1}.txt"), "w", encoding="utf-8") as f:
            f.write(full_transcript)
        print(f"Saved: ted_sample_{count+1}.wav, ted_sample_{count+1}.txt")

save_n_samples(tedlium_dataset, n=10, duration_sec=120, out_folder="data")

Saved: ted_sample_1.wav, ted_sample_1.txt
Saved: ted_sample_2.wav, ted_sample_2.txt
Saved: ted_sample_3.wav, ted_sample_3.txt
Saved: ted_sample_4.wav, ted_sample_4.txt
Saved: ted_sample_5.wav, ted_sample_5.txt
Saved: ted_sample_6.wav, ted_sample_6.txt
Saved: ted_sample_7.wav, ted_sample_7.txt
Saved: ted_sample_8.wav, ted_sample_8.txt
Saved: ted_sample_9.wav, ted_sample_9.txt
Saved: ted_sample_10.wav, ted_sample_10.txt


### Option 2: Audio books

Download 'train-clean-100.tar.gz [6.3G]' from LibriSpeech ASR corpus

 [Get gz file here](https://github.com/josephnguyen0413/02467_Assignment2)

### Extracting downdloaded gz file

In [16]:
import tarfile

# Open and extract the .tar.gz file
with tarfile.open('TEDLIUM_release1.tar.gz', 'r:gz') as tar:
    tar.extractall(path='TED_talks')  # you can set your target directory here


### Create audio file

In [ ]:
import os
import soundfile as sf
import numpy as np

# Set your input and output paths
input_folder = "audio_files/LibriSpeech/train-clean-100/32/21625"
output_file = "brownie_beaver_4_min.wav"

# Find and sort all .flac files in the folder
flac_files = sorted([
    f for f in os.listdir(input_folder)
    if f.lower().endswith(".flac")
])

if not flac_files:
    raise FileNotFoundError(f"No .flac files found in: {input_folder}")

# Combine audio
combined_audio = []
sample_rate = None

for flac_file in flac_files:
    path = os.path.join(input_folder, flac_file)
    audio_data, sr = sf.read(path)
    
    # Make sure sample rate matches
    if sample_rate is None:
        sample_rate = sr
    elif sr != sample_rate:
        raise ValueError(f"Sample rate mismatch in {flac_file}: {sr} != {sample_rate}")
    
    combined_audio.append(audio_data)

# Concatenate all audio data
final_audio = np.concatenate(combined_audio)

# Save as a .wav file
sf.write(output_file, final_audio, sample_rate)
print(f"Combined audio saved to: {output_file}")


Combined audio saved to: brownie_beaver_4_min.wav


### Getting transcipt for audio file

In [2]:
import os

# Paths
input_folder = "audio_files/LibriSpeech/train-clean-100/32/21625"
transcript_path = os.path.join(input_folder, "32-21625.trans.txt")
output_transcript = "brownie_beaver_4_min.txt"

# Load and parse the .trans file
transcript_map = {}

with open(transcript_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        parts = line.strip().split(" ", 1)
        if len(parts) == 2:
            key, text = parts
            transcript_map[key] = text

# Sort the keys to match audio order
sorted_keys = sorted(transcript_map.keys())

# Combine the transcriptions into one continuous line
combined_text = " ".join([transcript_map[k] for k in sorted_keys])

# Save to a text file
with open(output_transcript, "w", encoding="utf-8") as out_f:
    out_f.write(combined_text)

print(f" Combined transcript saved to: {output_transcript}")


 Combined transcript saved to: brownie_beaver_4_min.txt
